# Superdense coding with Qiskit

This notebook implements superdense coding and verifies all four possible two-bit messages.

In [ ]:
# Run this cell in Google Colab.
%pip install -q qiskit qiskit-aer

In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

In [3]:
def superdense_circuit(message):
    if len(message) != 2 or set(message) - {"0", "1"}:
        raise ValueError("the message must be one of 00, 01, 10, 11")

    phase_bit, value_bit = map(int, message)
    circuit = QuantumCircuit(2, 2)

    # Prepare and share a Bell pair.
    circuit.h(0)
    circuit.cx(0, 1)

    # Encode two classical bits in the sender's qubit.
    if value_bit:
        circuit.x(0)
    if phase_bit:
        circuit.z(0)

    # Decode after receiving the encoded qubit.
    circuit.cx(0, 1)
    circuit.h(0)

    # q0 contains the phase bit and q1 the value bit. Mapping them to
    # c1 and c0 makes Qiskit's displayed string equal to the message.
    circuit.measure([0, 1], [1, 0])
    return circuit

In [4]:
superdense_circuit("10").draw()

┌───┐     ┌───┐     ┌───┐┌─┐
q_0: ┤ H ├──■──┤ Z ├──■──┤ H ├┤M├
     └───┘┌─┴─┐└───┘┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├─────┤ X ├─┤M├──╫─
          └───┘     └───┘ └╥┘  ║ 
c: 2/══════════════════════╩═══╩═
                           0   1

In [5]:
def run_superdense(message, shots=1, seed=None):
    simulator = AerSimulator()
    compiled = transpile(superdense_circuit(message), simulator)
    result = simulator.run(compiled, shots=shots, seed_simulator=seed).result()
    return dict(result.get_counts())

In [6]:
for message in ["00", "01", "10", "11"]:
    counts = run_superdense(message, seed=14014)
    recovered = next(iter(counts))
    print(message, recovered)

00 00
01 01
10 10
11 11
